# Precision@5

전체 30개 질문을 읽고 dev 15개로 설정을 검토합니다.

## 실행 환경 준비

이 노트북은 제공 PDF에서 실제 텍스트를 추출합니다. 답변 생성 API는 호출하지 않습니다.

### 분모 정책을 코드에 고정하기

`search()`가 먼저 고유 문서 순위를 반환하고 `precision_at()`은 그 상위 `k`개 중 정답 ID 수를 약속한 `k`로 나눕니다. 중복 제거는 `precision_at()`의 책임이 아닙니다. 질문별 점수와 `returned_count`는 서로 다른 값입니다. 단일 정답 자료에서 Precision@5 최댓값은 0.2지만 여러 정답을 라벨링한 자료에서는 달라지므로 서로 다른 평가셋의 숫자를 그대로 비교하지 않습니다.

In [1]:
from pathlib import Path
import json, math, re
from collections import Counter
from pypdf import PdfReader

# 현재 폴더에서 저장소의 공개 RAG 자료를 찾습니다.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "labs/data/rag-corpus/manifest.json").exists())
corpus = root / "labs/data/rag-corpus"
dataset_path = root / "labs/part3/ch11-evaluation/data/evaluation-queries.jsonl"
dataset = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines()]
evaluation_items = [item for item in dataset if item["split"] == "dev"]
print(f"전체 질문: {len(dataset)}개")
print(f"설정 선택용 dev 질문: {len(evaluation_items)}개")

전체 질문: 30개
설정 선택용 dev 질문: 15개


## 공개 PDF를 페이지 단위 후보로 준비

스캔본은 OCR이 필요하므로 이번 기초 검색에서 제외합니다.

In [2]:
manifest = json.loads((corpus / "manifest.json").read_text(encoding="utf-8"))
pages = []
for item in manifest:
    if item["tier"] == "scanned":
        continue
    reader = PdfReader(corpus / item["file"])
    for page_no, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            pages.append({"doc_id": item["doc_id"], "page": page_no, "text": text})
print(f"검색 후보: {len(pages)}개 페이지")

검색 후보: 90개 페이지


## 같은 검색기를 모든 질문에 적용

단어와 한글 두 글자 조각의 겹침을 사용합니다. 점수가 같으면 문서 ID와 쪽 번호로 순서를 고정합니다.

In [3]:
def tokens(text):
    normalized = re.sub(r"\s+", " ", text.lower())
    words = re.findall(r"[가-힣a-z0-9]+", normalized)
    compact = re.sub(r"[^가-힣a-z0-9]", "", normalized)
    return words + [compact[i:i+2] for i in range(max(0, len(compact)-1))]

def search(question, limit=50):
    query_counts = Counter(tokens(question))
    scored = []
    for page in pages:
        page_counts = Counter(tokens(page["text"]))
        overlap = sum(min(count, page_counts[token]) for token, count in query_counts.items())
        score = overlap / math.sqrt(max(1, sum(page_counts.values())))
        scored.append({"doc_id": page["doc_id"], "page": page["page"], "score": score})
    ranked_pages = sorted(scored, key=lambda row: (-row["score"], row["doc_id"], row["page"]))
    unique_docs, seen = [], set()
    for row in ranked_pages:
        if row["doc_id"] not in seen:
            unique_docs.append(row)
            seen.add(row["doc_id"])
    return unique_docs[:limit]

sample = search(dataset[0]["question"], 5)
print([(row["doc_id"], row["page"], round(row["score"], 3)) for row in sample])

[('E01', 1, 0.926), ('M04', 3, 0.225), ('H03', 1, 0.188), ('H06', 5, 0.142), ('E02', 5, 0.132)]


## Precision@5 계산

`search()`가 이미 중복 문서 ID를 제거한 순위를 반환합니다. 그 상위 5개 문서 가운데 정답 문서가 차지하는 비율을 구합니다.

In [4]:
def precision_at(results, gold, k):
    return sum(row["doc_id"] in set(gold) for row in results[:k]) / k

rows = []
for item in evaluation_items:
    results = search(item["question"], 50)
    rows.append((item["qid"], precision_at(results, item["gold_doc_ids"], 5)))
mean_p5 = sum(score for _, score in rows) / len(rows)
print("Precision@5:", round(mean_p5, 3))
print("낮은 5개:", sorted(rows, key=lambda row: row[1])[:5])

Precision@5: 0.2
낮은 5개: [('Q001', 0.2), ('Q002', 0.2), ('Q003', 0.2), ('Q004', 0.2), ('Q005', 0.2)]


## k 값 비교

k가 커질수록 관련 없는 페이지가 포함될 여지가 늘어납니다.

In [5]:
for k in (1, 3, 5, 10):
    scores = [precision_at(search(item["question"], 50), item["gold_doc_ids"], k)
              for item in evaluation_items]
    print(f"Precision@{k}: {sum(scores) / len(scores):.3f}")

Precision@1: 0.733


Precision@3: 0.333


Precision@5: 0.200


Precision@10: 0.100
